# Natural Language Processing: an English-language introduction

**Learning outcomes:** distinguish NLP tasks, explain tokenization and ambiguity,
and define a binary sentiment-classification experiment.

| Task | English example | Output |
|---|---|---|
| Sentiment analysis | "The film was surprisingly good." | positive |
| Named-entity recognition | "Alex visited London." | person, location |
| Semantic similarity | "A lovely film" / "An excellent movie" | similarity score |
| Question answering | "Who directed the film?" | an answer span |
| Text generation | "Write a short review." | generated text |

In [1]:
import sys
from importlib.metadata import version
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
assert sys.version_info[:2] == (3, 12), "Select the Python 3.12 project kernel."
assert version('paddlepaddle').split("+")[0] == '3.3.1', "Select the locked project environment."
print("Python:", sys.version.split()[0])
print('paddlepaddle' + ":", version('paddlepaddle'))
print("Run mode:", 'Default classroom training parameters')

Python: 3.12.12
paddlepaddle: 3.3.1
Run mode: Default classroom training parameters


**Expected output:**

```text
Python: 3.12.12
paddlepaddle: 3.3.1
Run mode: Default classroom training parameters
```

## Why language is difficult

"Not bad" is not the same as "bad". "Great, another delay" may be sarcastic.
"The bank is closed" needs context. Word order, negation, domain, and document
length all affect meaning. Positive and negative are dataset labels, not a
complete description of every real review; mixed and neutral cases still exist.

English words are separated by spaces much of the time, but contractions,
punctuation, and names still need a tokenization policy. This course preserves
apostrophes and negation using a small regular-expression tokenizer. Production
transformer models use their own subword tokenizer instead.

In [2]:
from pathlib import Path
import os
import sys

# Find the English course root without relying on a specific working directory.
start = Path.cwd().resolve()
COURSE = next((p for p in [start, *start.parents]
               if (p / "COURSE_ROOT").is_file()), None)
if COURSE is None and (start / "english" / "COURSE_ROOT").is_file():
    COURSE = start / "english"
if COURSE is None:
    raise RuntimeError("Start Jupyter inside the project or the english directory.")
sys.path.insert(0, str(COURSE / "common"))
os.environ.setdefault("HF_HOME", str(COURSE / "data" / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(COURSE / "data" / "torch-cache"))

'/Users/will/workspace/deep-learning-lab/english/data/torch-cache'

**Expected output:**

```text
'…/deep-learning-lab/english/data/torch-cache'
```

(Inside the quoted output `…` stands for the repository's absolute path prefix on the machine that ran the notebook; yours will show your own path.)

In [3]:
from course_data import tokenize
examples = ["I loved the acting, but the ending was disappointing.",
            "The movie isn't bad at all.", "Great, another predictable ending."]
for text in examples:
    print("Text:", text)
    print("Tokens:", tokenize(text))

Text: I loved the acting, but the ending was disappointing.
Tokens: ['i', 'loved', 'the', 'acting', 'but', 'the', 'ending', 'was', 'disappointing']
Text: The movie isn't bad at all.
Tokens: ['the', 'movie', "isn't", 'bad', 'at', 'all']
Text: Great, another predictable ending.
Tokens: ['great', 'another', 'predictable', 'ending']


**Expected output:**

```text
Text: I loved the acting, but the ending was disappointing.
Tokens: ['i', 'loved', 'the', 'acting', 'but', 'the', 'ending', 'was', 'disappointing']
Text: The movie isn't bad at all.
Tokens: ['the', 'movie', "isn't", 'bad', 'at', 'all']
Text: Great, another predictable ending.
Tokens: ['great', 'another', 'predictable', 'ending']
```

## Our experiment

We use the [Stanford IMDb movie-review dataset](https://ai.stanford.edu/~amaas/data/sentiment/).
Labels are `negative=0` and `positive=1`. A validation split is drawn from the
official training data; the official test split is reserved for final evaluation.
The preparation code filters reviews containing CJK characters and removes exact
duplicate text across splits, so prepared counts can differ from published counts.

Vocabulary construction and Word2Vec fitting use training text only. Validation
and test words outside that vocabulary map to an unknown token. A pretrained
sentiment model is an optional separate comparison, not the LSTM being trained.

**Discuss:** How would you label a mixed review? Why should a negation word remain
in the input? Why is accuracy on a tiny classroom subset not a benchmark result?